In [1]:
import numpy as np
import sklearn
import torch
import os
from typing import OrderedDict

from torch.utils.data import TensorDataset, DataLoader

In [2]:
if not os.path.exists('tree_species_classifier_data.npz'):
  !wget -O tree_species_classifier_data.npz "https://www.dropbox.com/scl/fi/b7mw23k3ifaeui9m8nnn3/tree_species_classifier_data.npz?rlkey=bgxp37c1t04i7q35waf3slc26&dl=1"

In [3]:
data = np.load('tree_species_classifier_data.npz')
train_features = data['train_features']
train_labels = data['train_labels']
test_features = data['test_features']
test_labels = data['test_labels']

In [4]:
# examine dataset shape and train/test split
print(f'Train Features Shape: {train_features.shape} of type {train_features.dtype}')
print(f'Train Labels Shape: {train_labels.shape} of type {train_labels.dtype}')
print(f'Test Features Shape: {test_features.shape} of type {test_features.dtype}')
print(f'Test Labels Shape: {test_labels.shape} of type {test_labels.dtype}')

Train Features Shape: (15707, 426) of type int16
Train Labels Shape: (15707,) of type uint8
Test Features Shape: (1554, 426) of type int16
Test Labels Shape: (1554,) of type uint8


In [5]:
# PCA Pre-processing
pca = sklearn.decomposition.PCA(n_components=32, whiten=True)
pca.fit(train_features)

# apply the PCA to both train and test features
X_train_pca = pca.transform(train_features)
X_test_pca = pca.transform(test_features)

print(X_train_pca.shape)
print(X_test_pca.shape)

# create new pointers for naming convention's sake
y_train = train_labels
y_test = test_labels

(15707, 32)
(1554, 32)


In [6]:
# Linear classifier using sklearn
linear_model = sklearn.linear_model.LogisticRegression()
linear_model.fit(X_train_pca,y_train)
linear_model_accuracy = linear_model.score(X_test_pca, y_test)
print(f'The linear multi-class classifier has an accuracy of {linear_model_accuracy*100:.2f}%')


The linear multi-class classifier has an accuracy of 83.40%


In [7]:
# Neural Network using sklearn
nn = sklearn.neural_network.MLPClassifier(hidden_layer_sizes=(100,), random_state=1, max_iter=1000, learning_rate_init=0.0025, verbose=True, alpha=0.1)
nn.fit(X_train_pca, y_train)
nn_accuracy = nn.score(X_test_pca, y_test)
print(f'The neural network has an accuracy of {nn_accuracy*100:.2f}%')

Iteration 1, loss = 1.10198686
Iteration 2, loss = 0.56255615
Iteration 3, loss = 0.48344804
Iteration 4, loss = 0.44134234
Iteration 5, loss = 0.41237689
Iteration 6, loss = 0.39131708
Iteration 7, loss = 0.37250246
Iteration 8, loss = 0.36021761
Iteration 9, loss = 0.34807679
Iteration 10, loss = 0.33711313
Iteration 11, loss = 0.32878084
Iteration 12, loss = 0.32152963
Iteration 13, loss = 0.31636360
Iteration 14, loss = 0.31053654
Iteration 15, loss = 0.30333158
Iteration 16, loss = 0.29929284
Iteration 17, loss = 0.29425334
Iteration 18, loss = 0.29131487
Iteration 19, loss = 0.28878144
Iteration 20, loss = 0.28505136
Iteration 21, loss = 0.28102722
Iteration 22, loss = 0.27992803
Iteration 23, loss = 0.27592675
Iteration 24, loss = 0.27461242
Iteration 25, loss = 0.27198074
Iteration 26, loss = 0.27002863
Iteration 27, loss = 0.26663247
Iteration 28, loss = 0.26706586
Iteration 29, loss = 0.26430259
Iteration 30, loss = 0.26261709
Iteration 31, loss = 0.25998710
Iteration 32, los

When increasing the max number of iterations during the NN training, this only decreased the NN accuracy on the test set. This suggested that during the training process the NN was overfitting to the training data. Thus, I kept the number of max training iterations high but significantly increased the alpha parameter in order to increase the magnitude of regularization in an attempt to reduce the higher order terms' coefficients and thus reduce overfitting, which seemed to work, boosting the accuracy above the linear model's, being about 85% and 83% respectively.

In [8]:
# Classifier using PyTorch
# turn dataset into tensors
X_train_pca_T = torch.tensor(X_train_pca).float()
X_test_pca_T = torch.tensor(X_test_pca).float()
y_train_T = torch.tensor(y_train)
y_test_T = torch.tensor(y_test)

# load data into torch data handler objects
trainset = TensorDataset(X_train_pca_T, y_train_T)
trainloader = DataLoader(trainset, batch_size=32, shuffle=True)

testset = TensorDataset(X_test_pca_T, y_test_T)
testloader = DataLoader(testset, batch_size=32, shuffle=False)

In [9]:
def calc_torch_accuracy(model: torch.nn.Sequential, testloader: DataLoader) -> float:
    iter_testloader = iter(testloader)  # turn DataLoader into an iterable
    n_correct: int = 0

    while(True):
        try:
            batch = next(iter_testloader)
            batch_features, batch_labels = batch
            y_predict = torch.argmax(model.forward(batch_features), dim=1)
            n_correct += torch.sum(y_predict == batch_labels)
        except StopIteration:
            break

    n: int = len(testloader.dataset.tensors[0])
    return float(n_correct / n)

In [14]:
def train_torch_nn(model: torch.nn.Sequential, trainloader: DataLoader, testloader: DataLoader, loss_fn: torch.nn.CrossEntropyLoss, optimizer: torch.optim.SGD, epochs: int=100) -> None:
    X, y = trainloader.dataset.tensors

    for i in range(epochs):
        optimizer.zero_grad()
        outputs = model(X)
        loss = loss_fn(outputs, y)
        loss.backward()
        optimizer.step()

        # calc and display accuracies
        train_acc = calc_torch_accuracy(model, trainloader)
        test_acc = calc_torch_accuracy(model, testloader)

        print(f'Epoch {i:4d} - Training Accuracy: {train_acc:.6f}, Test Accuracy: {test_acc:.6f}')

In [15]:
input_size = X_train_pca.shape[1]
output_size = max(y_train)+1  # number of classes

In [ ]:
# PyTorch Linear Classifier Implementation
torch_lin = torch.nn.Sequential(
    torch.nn.Linear(input_size, output_size)
)

lin_loss_fn = torch.nn.CrossEntropyLoss()
lin_optimizer = torch.optim.SGD(torch_lin.parameters(), lr=0.01, weight_decay=0.001)

torch_lin.train()
train_torch_nn(
    model=torch_lin,
    trainloader=trainloader,
    testloader=testloader,
    loss_fn=lin_loss_fn,
    optimizer=lin_optimizer,
    epochs=1000
)

Epoch    0 - Training Accuracy: 0.124403, Test Accuracy: 0.142857
Epoch    1 - Training Accuracy: 0.125931, Test Accuracy: 0.144788
Epoch    2 - Training Accuracy: 0.128032, Test Accuracy: 0.146718
Epoch    3 - Training Accuracy: 0.129624, Test Accuracy: 0.148649
Epoch    4 - Training Accuracy: 0.131406, Test Accuracy: 0.150579
Epoch    5 - Training Accuracy: 0.133316, Test Accuracy: 0.151223
Epoch    6 - Training Accuracy: 0.135672, Test Accuracy: 0.153797
Epoch    7 - Training Accuracy: 0.137455, Test Accuracy: 0.157014
Epoch    8 - Training Accuracy: 0.139810, Test Accuracy: 0.160232
Epoch    9 - Training Accuracy: 0.141911, Test Accuracy: 0.162162
Epoch   10 - Training Accuracy: 0.143057, Test Accuracy: 0.165380
Epoch   11 - Training Accuracy: 0.144776, Test Accuracy: 0.167954
Epoch   12 - Training Accuracy: 0.147259, Test Accuracy: 0.169884
Epoch   13 - Training Accuracy: 0.149615, Test Accuracy: 0.173102
Epoch   14 - Training Accuracy: 0.150952, Test Accuracy: 0.176963
Epoch   15

In [ ]:
# PyTorch NN Implementation
hidden_size = 100

torch_nn = torch.nn.Sequential(
                torch.nn.Linear(input_size, hidden_size),
                torch.nn.ReLU(),
                torch.nn.Linear(hidden_size, output_size)
    )
print(torch_nn)

nn_loss_fn = torch.nn.CrossEntropyLoss()
nn_optimizer = torch.optim.SGD(torch_nn.parameters(), lr=0.01, weight_decay=0.001)

torch_nn.train()
train_torch_nn(
    model=torch_nn,
    trainloader=trainloader,
    testloader=testloader,
    loss_fn=nn_loss_fn,
    optimizer=nn_optimizer,
    epochs=1000
)

Sequential(
  (0): Linear(in_features=32, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=8, bias=True)
)
Epoch    0 - Training Accuracy: 0.130579, Test Accuracy: 0.118404
Epoch    1 - Training Accuracy: 0.133316, Test Accuracy: 0.122909
Epoch    2 - Training Accuracy: 0.136627, Test Accuracy: 0.123552
Epoch    3 - Training Accuracy: 0.139619, Test Accuracy: 0.128057
Epoch    4 - Training Accuracy: 0.141784, Test Accuracy: 0.131274
Epoch    5 - Training Accuracy: 0.144904, Test Accuracy: 0.138353
Epoch    6 - Training Accuracy: 0.147896, Test Accuracy: 0.142214
Epoch    7 - Training Accuracy: 0.151461, Test Accuracy: 0.147362
Epoch    8 - Training Accuracy: 0.154963, Test Accuracy: 0.150579
Epoch    9 - Training Accuracy: 0.159483, Test Accuracy: 0.157014
Epoch   10 - Training Accuracy: 0.162157, Test Accuracy: 0.159588
Epoch   11 - Training Accuracy: 0.165595, Test Accuracy: 0.162162
Epoch   12 - Training Accuracy: 0.168396, Test Accuracy: 0.1608

### Code Explanation
Sources: I mostly used the sklearn and pytorch documentation/APIs for guidance when completing this assignment, especially the examples for how the different classes and functions I needed were used, as well as which classes and functions were even available. For example, using the sklearn docs to understand the difference between PCA.fit() and PCA.transform, or looking at the examples in the torch.nn.Sequential() docs when implementing my models with PyTorch. That being said, I took a glance at Professor Ventura's paper on this topic and the linked Github with the code used in order to double check that I was applying my PCA correctly, as the dimensions of the matrix returned by the PCA.fit() function initially caused me some confusion. I also used Perplexity (AI) for double checking how to use torch.nn.Sequential() for just a linear multiclass classifier, and further explanation of several things like what loss_fn.backward() is doing under the hood, why the PyTorch docs uses iter() in its DataLoader usage examples, among other things.

### Discussion
Each row correponds to a data point, while the columns correspond to the data points' features. The ranges of the labels are [0, 7], corresponding the the 8 classes in this classification problem, each of which correspond to a possible tree species.